In [ ]:
# ============================================================
# INPUT SYMBOLS AS A LIST WITH ANCHOR SYMBOL LAST
# ============================================================

sorted_symbols_list = ['CWB', 'ICVT']

moving_avg_days = 20

start_date = 
end_date = 

go_long_hurdle = -.0025
exit_long_hurdle = 0


In [6]:
# --- system setup ---
import sys
import os
from pathlib import Path 
sys.path.append(os.path.abspath(".."))

# from datetime import date
import asyncio
import numpy as np
import pandas as pd


In [7]:
filename = "_".join(sorted_symbols_list)
filename = f"{filename}_{moving_avg_days}.csv"
file_path = Path("unit_prices/") / filename

df = pd.read_csv(file_path, index_col=0)

In [9]:
df["unit price pct diff"] = (
    df["unit price pct diff"]
    .str.rstrip("%")
    .astype(float)
    / 100
)

In [10]:
df['go long'] = df['unit price pct diff'] < go_long_hurdle
df['go short'] = df['unit price pct diff'] > go_short_hurdle
df['exit long'] = df['unit price pct diff'] > exit_long_hurdle
df['exit short'] = df['unit price pct diff'] < exit_short_hurdle

df["current_position"] = 0
df["new_position"] = 0

current_col = df.columns.get_loc("current_position")
new_col     = df.columns.get_loc("new_position")

for i in range(1, len(df)):
    current_pos = df.iat[i - 1, new_col]
    new_pos = current_pos

    if current_pos == 0:
        new_pos = 1 if df["go long"].iat[i] else -1 if df["go short"].iat[i] else 0

    elif current_pos == 1 and df["exit long"].iat[i]:
        new_pos = 0

    elif current_pos == -1 and df["exit short"].iat[i]:
        new_pos = 0

    df.iat[i, current_col] = current_pos
    df.iat[i, new_col] = new_pos


In [11]:
# filename = filename.replace(".csv", f"_{moving_avg_days}.csv")

file_path = Path("with positions/") / filename

df.to_csv(file_path)

print()
print('finished')
print()


finished

